# Hello, library — several topics behind one handle

A `library()` is a folder of topic stores. You can ask one topic directly, let the
library `route()` a question to the topics that look relevant, or `ask()` across all
of them at once.

**Needs:** Ollama with `nomic-embed-text`.

In [1]:
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

from slim_llm_memory import library

db = library(ROOT / ".hello_nb" / "library")

db.topic("webserver").add({
    "nginx.md": "To serve a site with nginx, put a server block in /etc/nginx/sites-enabled "
                "and run `nginx -s reload`.",
    "tls.md": "Renew the TLS certificate with `certbot renew`. It writes fresh pems into "
              "/etc/letsencrypt/live and reloads nginx afterwards.",
})
db.topic("kitchen").add({
    "carbonara.md": "Carbonara is guanciale, pecorino, egg yolks and pepper — never cream.",
    "risotto.md": "Risotto wants hot stock added one ladle at a time, and constant stirring "
                  "so the rice releases its starch.",
})

db.topics()

[kitchen: 2 doc(s), 2 chunks, webserver: 2 doc(s), 2 chunks]

## Routing

`route()` scores the question against each topic's centroid and returns the ones worth
searching. This is the cheap step: no chunk is read, only one vector per topic.

In [2]:
db.route("how do I renew the certificate?")

route('how do I renew the certificate?')  → ['webserver']  · embed 2043 ms · route 0.28 ms
  0.65  webserver
  0.42  kitchen

## Asking

`ask()` routes first, then searches only the topics that survived. The hits carry which
topic they came from.

In [3]:
r = db.ask("how do I renew the certificate?")
r

ask('how do I renew the certificate?')  4 hit(s) · hybrid · embed 2245 ms · scan 2.29 ms
   1  0.72  webserver/tls.md#0       Renew the TLS certificate with `certbot renew`. It writes fres  [both]
   2  0.44  webserver/nginx.md#0     To serve a site with nginx, put a server block in /etc/nginx/s  [dense]
   3  0.39  kitchen/risotto.md#0     Risotto wants hot stock added one ladle at a time, and constan  [dense]
   4  0.34  kitchen/carbonara.md#0   Carbonara is guanciale, pecorino, egg yolks and pepper — never  [dense]

A question from the other side of the library lands in the other topic:

In [4]:
db.ask("what goes in carbonara?")

ask('what goes in carbonara?')  4 hit(s) · hybrid · embed 2200 ms · scan 0.35 ms
   1  0.73  kitchen/carbonara.md#0   Carbonara is guanciale, pecorino, egg yolks and pepper — never  [both]
   2  0.46  kitchen/risotto.md#0     Risotto wants hot stock added one ladle at a time, and constan  [dense]
   3  0.38  webserver/nginx.md#0     To serve a site with nginx, put a server block in /etc/nginx/s  [dense]
   4  0.35  webserver/tls.md#0       Renew the TLS certificate with `certbot renew`. It writes fres  [dense]

In [5]:
db.close()